# 04 - Fine-tune Mushkil (AraT5v2)

**Base Model:** `riotu-lab/mushkil` (already fine-tuned AraT5v2)
**Architecture:** AraT5v2 - Arabic T5 Seq2Seq
**Approach:** Treats diacritization as translation task

**Techniques:**
- Further fine-tuning on KSSA dataset
- Curriculum Learning (short to long texts)
- Label Smoothing
- Mixed Precision (FP16)
- Gradient Checkpointing

In [1]:
!pip install -q transformers torch accelerate tqdm datasets sentencepiece

In [2]:
# Setup
import os, sys, json, re, torch, gc, subprocess, zipfile
from datetime import datetime
from tqdm import tqdm
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

IS_RUNPOD = False  # Cloud detection removed
PROJECT_DIR = Path('.')
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'notebooks'))

# Import utilities (includes safeguards for empty responses & separated characters)
from utils_diacritization import (
    normalize_arabic, remove_diacritics, postprocess_diacritics,
    is_valid_output, repair_output, get_safe_seq2seq_generation_config
)

DATA_DIR = PROJECT_DIR / 'Public_Data_TrainDev'
TRAIN_DIR = DATA_DIR / 'Train'
DEV_INPUT = DATA_DIR / 'dev input-output' / 'Dev_input.json'
DEV_OUTPUT = DATA_DIR / 'dev input-output' / 'Dev_output.json'
TEST_DIR = PROJECT_DIR / 'Test'
TEST_INPUT = TEST_DIR / 'test_input.json'
OUTPUT_DIR = PROJECT_DIR / 'outputs'
SUBMISSION_DIR = PROJECT_DIR / 'submissions'
CHECKPOINTS_DIR = PROJECT_DIR / 'checkpoints' / 'mushkil_ft'
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
SUBMISSION_DIR.mkdir(exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Environment: 'Local' | Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB)")

def clear_gpu():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()
        print(f"GPU Memory: {torch.cuda.memory_allocated()/1024**3:.2f} GB allocated")

Environment: Local | Device: cuda
GPU: NVIDIA RTX A5000 (23.6 GB)


In [6]:
# Load Training Data
import pandas as pd

train_tsv = TRAIN_DIR / 'train_list.tsv'
train_df = pd.read_csv(train_tsv, sep='\t')
print(f"Training samples in TSV: {len(train_df)}")

def load_training_data():
    data = []
    text_dir = TRAIN_DIR / 'train_text'
    
    for idx, row in tqdm(train_df.iterrows(), total=len(train_df), desc="Loading data"):
        txt_file = text_dir / f"{row['stem']}.txt"
        if txt_file.exists():
            with open(txt_file, 'r', encoding='utf-8') as f:
                diacritized = f.read().strip()
            # Remove diacritics to get undiacritized version
            undiacritized = re.sub(r'[\u064B-\u0652]', '', diacritized)
            data.append({
                'id': row['stem'],
                'text_undiacritized': undiacritized,
                'text_diacritized': diacritized
            })
    return data

train_data = load_training_data()
# Curriculum learning: sort by length (easy to hard)
train_data = sorted(train_data, key=lambda x: len(x['text_undiacritized']))
print(f"Loaded {len(train_data)} training samples")

Training samples in TSV: 2327


Loading data: 100%|██████████| 2327/2327 [00:08<00:00, 275.16it/s]

Loaded 2327 training samples


In [5]:
# Load the ALREADY fine-tuned Mushkil model
from transformers import (
    AutoModelForSeq2SeqLM, T5Tokenizer,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)
from datasets import Dataset
from huggingface_hub import hf_hub_download

MODEL_NAME = 'riotu-lab/mushkil'  # Already fine-tuned AraT5v2
MODEL_KEY = 'mushkil_ft'  # ft = further fine-tuned

# Load tokenizer from SentencePiece only (tokenizer.json causes KeyError in newer transformers)
print(f"Loading {MODEL_NAME} (already fine-tuned AraT5v2)...")
spiece_path = hf_hub_download(MODEL_NAME, "spiece.model")
tokenizer = T5Tokenizer(vocab_file=spiece_path)
# Load in float32 for training; Trainer handles mixed precision
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
).to(device)

model.gradient_checkpointing_enable()
print(f"Model loaded: {sum(p.numel() for p in model.parameters()):,} parameters")

Loading riotu-lab/mushkil (already fine-tuned AraT5v2)...


spiece.model:   0%|          | 0.00/2.35M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/835 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

Model loaded: 367,508,736 parameters


In [7]:
# Prepare Dataset
MAX_LENGTH = 512

def preprocess_function(examples):
    model_inputs = tokenizer(
        examples['text_undiacritized'], max_length=MAX_LENGTH, truncation=True, padding='max_length'
    )
    labels = tokenizer(
        examples['text_diacritized'], max_length=MAX_LENGTH, truncation=True, padding='max_length'
    )
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

train_dataset = Dataset.from_list(train_data)
train_dataset = train_dataset.map(preprocess_function, batched=True, remove_columns=['id', 'text_undiacritized', 'text_diacritized'])

# Load dev data for validation
with open(DEV_INPUT, 'r', encoding='utf-8') as f: dev_input = json.load(f)
with open(DEV_OUTPUT, 'r', encoding='utf-8') as f: dev_output = json.load(f)
dev_lookup = {item['id']: item['text_diacritized'] for item in dev_output}
dev_data = [{'text_undiacritized': item['text_undiacritized'], 
             'text_diacritized': dev_lookup.get(item['id'], '')} 
            for item in dev_input if item['id'] in dev_lookup]

eval_dataset = Dataset.from_list(dev_data)
eval_dataset = eval_dataset.map(preprocess_function, batched=True, remove_columns=['text_undiacritized', 'text_diacritized'])

print(f"Train: {len(train_dataset)}, Eval: {len(eval_dataset)}")

Map:   0%|          | 0/2327 [00:00<?, ? examples/s]

Map:   0%|          | 0/260 [00:00<?, ? examples/s]

Train: 2327, Eval: 260


In [8]:
# Training Arguments - Optimized for 24GB GPU
training_args = Seq2SeqTrainingArguments(
    output_dir=str(CHECKPOINTS_DIR),
    num_train_epochs=3,
    per_device_train_batch_size=8,  # Increased from 4
    per_device_eval_batch_size=8,   # Increased from 4
    gradient_accumulation_steps=4,   # Reduced to compensate (effective batch: 32)
    learning_rate=1e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=True,  # Enable FP16 for faster training
    bf16=False,
    label_smoothing_factor=0.1,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,
    load_best_model_at_end=True,
    logging_steps=50,
    report_to="none",
    predict_with_generate=True,
    generation_max_length=MAX_LENGTH,
    dataloader_num_workers=4,  # Parallel data loading
    dataloader_pin_memory=True,
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)
print(f"Training config: batch={training_args.per_device_train_batch_size}, accum={training_args.gradient_accumulation_steps}, effective={training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Training config: batch=8, accum=4, effective=32


In [9]:
# Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

# Train with checkpoint resume
clear_gpu()
checkpoint = None
checkpoints = list(CHECKPOINTS_DIR.glob('checkpoint-*'))
if checkpoints:
    checkpoint = str(max(checkpoints, key=lambda x: int(x.name.split('-')[1])))
    print(f"Resuming from: {checkpoint}")

trainer.train(resume_from_checkpoint=checkpoint)

GPU Memory: 1.37 GB allocated


Step,Training Loss,Validation Loss
200,35.943914,8.875390


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


TrainOutput(global_step=219, training_loss=43.573608119738154, metrics={'train_runtime': 676.4948, 'train_samples_per_second': 10.319, 'train_steps_per_second': 0.324, 'total_flos': 6066287836397568.0, 'train_loss': 43.573608119738154, 'epoch': 3.0})

In [10]:
# Save
final_path = CHECKPOINTS_DIR / 'final'
trainer.save_model(str(final_path))
tokenizer.save_pretrained(str(final_path))
print(f"Saved to: {final_path}")
clear_gpu()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to: ./checkpoints/mushkil_ft/final
GPU Memory: 4.13 GB allocated


In [11]:
clear_gpu()

GPU Memory: 4.13 GB allocated


## Inference on Dev and Test Sets

In [12]:
# Load fine-tuned model for inference (FP16 for speed)
final_model_path = CHECKPOINTS_DIR / 'final'
if final_model_path.exists():
    print(f"Loading fine-tuned model from {final_model_path}")
    # Clear training model from memory first
    del model
    clear_gpu()
    
    model = AutoModelForSeq2SeqLM.from_pretrained(
        final_model_path,
        torch_dtype=torch.float16,  # FP16 for faster inference
        device_map="auto"
    )
    tokenizer = T5Tokenizer.from_pretrained(final_model_path)
else:
    print("Using model from training (already in memory)")
    model = model.half()  # Convert to FP16 for inference

model.eval()
print("Model ready for inference (FP16)")

Loading fine-tuned model from ./checkpoints/mushkil_ft/final
GPU Memory: 4.13 GB allocated


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model ready for inference (FP16)


In [13]:
@torch.inference_mode()
def diacritize(text: str) -> str:
    """
    Diacritize Arabic text with safeguards against empty responses and separated characters.
    """
    original_text = text
    text = normalize_arabic(text)
    
    inputs = tokenizer(text, return_tensors="pt", max_length=MAX_LENGTH, truncation=True).to(device)
    
    # Get safe generation config
    gen_config = get_safe_seq2seq_generation_config()
    gen_config['max_length'] = MAX_LENGTH
    
    outputs = model.generate(**inputs, **gen_config)
    result = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
    
    # Repair output (fixes separated characters, validates, falls back to input if invalid)
    result = repair_output(result, original_text, fallback_to_input=True)
    
    return result

In [14]:
# Dev Set
CHECKPOINT_FILE = OUTPUT_DIR / f"{MODEL_KEY}_checkpoint.json"

def load_checkpoint():
    if CHECKPOINT_FILE.exists():
        with open(CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
            data = json.load(f)
            return set(data.get('processed_ids', [])), data.get('predictions', [])
    return set(), []

def save_checkpoint(predictions):
    with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump({'processed_ids': [p['id'] for p in predictions], 'predictions': predictions}, f, ensure_ascii=False)

processed_ids, dev_predictions = load_checkpoint()
print(f"Dev: {len(dev_input)} samples, {len(processed_ids)} already done")

for item in tqdm(dev_input, desc="Dev Set"):
    if item['id'] in processed_ids: continue
    try:
        result = diacritize(item['text_undiacritized'])
        dev_predictions.append({'id': item['id'], 'text_diacritized': result})
        save_checkpoint(dev_predictions)
    except Exception as e:
        dev_predictions.append({'id': item['id'], 'text_diacritized': item['text_undiacritized']})
        save_checkpoint(dev_predictions)

with open(OUTPUT_DIR / f"{MODEL_KEY}_dev_predictions.json", 'w', encoding='utf-8') as f:
    json.dump(dev_predictions, f, ensure_ascii=False, indent=2)

Dev: 260 samples, 0 already done


Dev Set: 100%|██████████| 260/260 [01:49<00:00,  2.38it/s]


In [15]:
# Compute Dev Metrics
DIACRITIC_PATTERN = re.compile(r'[\u064B-\u0652]')
ARABIC_LETTER_PATTERN = re.compile(r'[\u0621-\u064A]')

def extract_diacritics(text):
    result = []
    i = 0
    while i < len(text):
        if ARABIC_LETTER_PATTERN.match(text[i]):
            diacritics = []
            j = i + 1
            while j < len(text) and DIACRITIC_PATTERN.match(text[j]):
                diacritics.append(text[j])
                j += 1
            result.append((text[i], ''.join(diacritics)))
            i = j
        else:
            i += 1
    return result

def compute_metrics(predictions, ground_truth):
    gt_lookup = {item['id']: item['text_diacritized'] for item in ground_truth}
    total_chars, total_words, char_errors, word_errors, n = 0, 0, 0, 0, 0
    for pred in predictions:
        if pred['id'] not in gt_lookup: continue
        pred_pairs = extract_diacritics(pred['text_diacritized'])
        ref_pairs = extract_diacritics(gt_lookup[pred['id']])
        for i in range(min(len(pred_pairs), len(ref_pairs))):
            if pred_pairs[i][1] != ref_pairs[i][1]: char_errors += 1
        char_errors += abs(len(pred_pairs) - len(ref_pairs))
        total_chars += max(len(pred_pairs), len(ref_pairs))
        pred_words, ref_words = pred['text_diacritized'].split(), gt_lookup[pred['id']].split()
        for i in range(min(len(pred_words), len(ref_words))):
            if pred_words[i] != ref_words[i]: word_errors += 1
        word_errors += abs(len(pred_words) - len(ref_words))
        total_words += max(len(pred_words), len(ref_words))
        n += 1
    return {'DER': char_errors/total_chars if total_chars else 0, 'WER': word_errors/total_words if total_words else 0}

metrics = compute_metrics(dev_predictions, dev_output)
print(f"\nDEV RESULTS: DER={metrics['DER']*100:.2f}% | WER={metrics['WER']*100:.2f}%")


DEV RESULTS: DER=78.76% | WER=97.48%


In [16]:
# Test Set
with open(TEST_INPUT, 'r', encoding='utf-8') as f: test_input = json.load(f)
TEST_CHECKPOINT_FILE = OUTPUT_DIR / f"{MODEL_KEY}_test_checkpoint.json"

def load_test_checkpoint():
    if TEST_CHECKPOINT_FILE.exists():
        with open(TEST_CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
            data = json.load(f)
            return set(data.get('processed_ids', [])), data.get('predictions', [])
    return set(), []

def save_test_checkpoint(predictions):
    with open(TEST_CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump({'processed_ids': [p['id'] for p in predictions], 'predictions': predictions}, f, ensure_ascii=False)

test_processed_ids, test_predictions = load_test_checkpoint()
print(f"Test: {len(test_input)} samples, {len(test_processed_ids)} already done")

for item in tqdm(test_input, desc="Test Set"):
    if item['id'] in test_processed_ids: continue
    try:
        result = diacritize(item['text_undiacritized'])
        test_predictions.append({'id': item['id'], 'text_diacritized': result})
        save_test_checkpoint(test_predictions)
    except:
        test_predictions.append({'id': item['id'], 'text_diacritized': item['text_undiacritized']})
        save_test_checkpoint(test_predictions)

# Submission
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
json_file = SUBMISSION_DIR / f"{MODEL_KEY}_submission.json"
with open(json_file, 'w', encoding='utf-8') as f:
    json.dump(test_predictions, f, ensure_ascii=False, indent=2)
zip_file = SUBMISSION_DIR / f"{MODEL_KEY}_submission_{timestamp}.zip"
with zipfile.ZipFile(zip_file, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(json_file, 'submission.json')
print(f"SUBMISSION: {zip_file}")

Test: 328 samples, 0 already done


Test Set: 100%|██████████| 328/328 [02:03<00:00,  2.65it/s]

SUBMISSION: ./submissions/mushkil_ft_submission_20260224_075546.zip


In [17]:
# Sync and cleanup
sync_script = PROJECT_DIR / 'sync_results.sh'
if sync_script.exists() and False:
    subprocess.run(['bash', str(sync_script)])

# Final cleanup - free GPU memory
print("\nCleaning up GPU memory...")
del model
del tokenizer
clear_gpu()
print("Done! GPU memory released.")

KSAA 2026 - Sync Results to Google Drive
  Project: .

[1/3] Syncing outputs...
Transferred:   	          0 B / 156.451 KiB, 0%, 0 B/s, ETA -
Checks:                28 / 28, 100%
Transferred:            0 / 4, 0%
Elapsed time:         0.9s
Transferring:
 *                    mushkil_ft_checkpoint.json:  0% /46.975Ki, 0/s, -
 *               mushkil_ft_dev_predictions.json:  0% /44.244Ki, 0/s, -
 *               mushkil_ft_test_checkpoint.json:  0% /59.253Ki, 0/s, -
 *              tashkeel_700m_ft_checkpoint.json:  0% /5.979Ki, 0/s, -
2026-02-24 07:55:48 ERROR : tashkeel_700m_ft_checkpoint.json: Failed to copy: Post "https://www.googleapis.com/upload/drive/v3/files?alt=json&fields=id%2Cname%2Csize%2Cmd5Checksum%2Ctrashed%2CexplicitlyTrashed%2CmodifiedTime%2CcreatedTime%2CmimeType%2Cparents%2CwebViewLink%2CshortcutDetails%2CexportLinks&keepRevisionForever=false&prettyPrint=false&supportsAllDrives=true&uploadType=multipart": googleapi: Copy failed: can't copy - source file is being updat

2026/02/24 07:55:53 Failed to sync: Post "https://www.googleapis.com/upload/drive/v3/files?alt=json&fields=id%2Cname%2Csize%2Cmd5Checksum%2Ctrashed%2CexplicitlyTrashed%2CmodifiedTime%2CcreatedTime%2CmimeType%2Cparents%2CwebViewLink%2CshortcutDetails%2CexportLinks&keepRevisionForever=false&prettyPrint=false&supportsAllDrives=true&uploadType=multipart": googleapi: Copy failed: can't copy - source file is being updated (size changed from 7490 to 7754)


GPU Memory: 4.13 GB allocated
Done! GPU memory released.
